In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
# 소수점 아래 10자리까지 출력
pd.options.display.float_format = '{:.10f}'.format

In [ ]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d", "jacobi1d", "fastwalshtransform", "relu", "fft", "fir", 
    "sssp", "convolution2d", "color", "bitonicsort", "pagerank", "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'mm2', 
    'mm3', 
    'gemm',
    'gaussian',
    'gramschmidt',
    'vgg16',
    'spmv',
    'mis',
    'stencil2d',
    'lu',
    'matrixmultiplication',
    'matrixtranspose',
    'im2col',
    'nbody',
    'bfs',
]

In [17]:
low_sharing_benchmarks = [
    "fdtd2d", "jacobi1d", "fastwalshtransform", "relu", "fft", "fir", 
    "sssp", "convolution2d", "color", "bitonicsort", "pagerank", "simpleconvolution",
]

high_sharing_benchmarks = [
    'gups',    
    'syr2k',
    'mm2', 
    'mm3', 
    'gemm',
    'gaussian',
    'gramschmidt',
    'vgg16',
    'spmv',
    'mis',
    'stencil2d',
    'lu',
    'matrixmultiplication',
    'matrixtranspose',
    'im2col',
    'nbody',
    'bfs',
]

In [18]:
latency = {
    "low_avg": [
        277.0161, 191.0630, 266.6053, 343.7781, 421.7362, 
        632.9585, 580.5830, 659.4286, 749.6807, 861.2695
    ],
    "high_avg": [
        528.3369, 724.4577, 855.9928, 1096.4780, 1078.8750, 
        1270.3450, 1266.1070, 1250.0670, 1363.8490, 1446.0620
    ]
}

In [6]:
num_chiplets = 36
curr_chip_cols = [str(i) for i in range(36)]

clusters_3x3 = {
    'cluster_0_3x3': [0, 1, 2, 6, 7, 8, 12, 13, 14],
    'cluster_1_3x3': [3, 4, 5, 9, 10, 11, 15, 16, 17],
    'cluster_2_3x3': [18, 19, 20, 24, 25, 26, 30, 31, 32],
    'cluster_3_3x3': [21, 22, 23, 27, 28, 29, 33, 34, 35],
}

clusters_2x2 = {
    'cluster_0_2x2': [0, 1, 6, 7],       # 좌측 상단
    'cluster_1_2x2': [2, 3, 8, 9],       # 상단 중앙
    'cluster_2_2x2': [4, 5, 10, 11],     # 우측 상단
    'cluster_3_2x2': [12, 13, 18, 19],   # 좌측 중앙
    'cluster_4_2x2': [14, 15, 20, 21],   # 정중앙
    'cluster_5_2x2': [16, 17, 22, 23],   # 우측 중앙
    'cluster_6_2x2': [24, 25, 30, 31],   # 좌측 하단
    'cluster_7_2x2': [26, 27, 32, 33],   # 하단 중앙
    'cluster_8_2x2': [28, 29, 34, 35]    # 우측 하단
}

clusters_3x3_cols = [k for k in clusters_3x3.keys()]
clusters_2x2_cols = [k for k in clusters_2x2.keys()]

chiplet_to_cluster_2x2 = {}
for cluster_name, chiplet_ids in clusters_2x2.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_2x2[chip_id] = cluster_name

chiplet_to_cluster_3x3 = {}
for cluster_name, chiplet_ids in clusters_3x3.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_3x3[chip_id] = cluster_name

In [7]:
def get_cluster_id(id):
    row = id // 6
    col = id % 6

    partRow = row // 3
    partCol = col // 3

    return partRow * 2 + partCol

In [8]:
def get_detailed_df(df):
    # 1. 클러스터별 access 계산 (원본 로직 유지)
    for cluster_name, chiplet_ids in clusters_2x2.items():
        str_cols = [str(x) for x in chiplet_ids]
        df[cluster_name] = df[str_cols].sum(axis=1)

    for cluster_name, chiplet_ids in clusters_3x3.items():
        str_cols = [str(x) for x in chiplet_ids]
        df[cluster_name] = df[str_cols].sum(axis=1)

    # 3. addr 단위로 묶어서 Sharing 판별 (interval 제거)
    agg_dict = {col: 'sum' for col in curr_chip_cols}
    agg_dict.update({col: 'sum' for col in clusters_2x2})
    agg_dict.update({col: 'sum' for col in clusters_3x3})
    agg_dict['rw'] = 'max'
    agg_dict['owner'] = 'first'
    
    # 핵심: interval 없이 addr로만 그룹화
    addr_df = df.groupby('addr', as_index=False).agg(agg_dict)

    # 2x2, 3x3 클러스터 별 owner 설정
    addr_df['owner_cluster_2x2'] = addr_df['owner'].map(chiplet_to_cluster_2x2)
    addr_df['owner_cluster_3x3'] = addr_df['owner'].map(chiplet_to_cluster_3x3)

    for i in range(num_chiplets):
        col = str(i)
        addr_df[col] = np.where(addr_df['owner'] == i, 0, addr_df[col])

    for col in clusters_2x2_cols:
        addr_df[col] = np.where(addr_df['owner_cluster_2x2'] == col, 0, addr_df[col])

    for col in clusters_3x3_cols:
        addr_df[col] = np.where(addr_df['owner_cluster_3x3'] == col, 0, addr_df[col])

    # Sharing 여부 및 Degree 계산
    addr_df['is_shared'] = (addr_df[curr_chip_cols] != 0).any(axis=1)
    addr_df['shared_degree'] = (addr_df[curr_chip_cols] != 0).sum(axis=1)
    addr_df['shared_degree_2x2'] = (addr_df[list(clusters_2x2.keys())] != 0).sum(axis=1)
    addr_df['shared_degree_3x3'] = (addr_df[list(clusters_3x3.keys())] != 0).sum(axis=1)

    addr_df['is_rw'] = addr_df['rw'] == 1
    addr_df['is_ro'] = addr_df['rw'] == 0

    # 4. 판별된 상태를 원본 데이터에 맵핑
    flags = addr_df[['addr', 'is_shared', 'is_rw', 'is_ro', 
                     'shared_degree', 'shared_degree_2x2', 'shared_degree_3x3']]
    
    # [수정] 만약 df에 이미 같은 이름의 컬럼이 있다면 삭제 후 병합 (중복 방지)
    cols_to_drop = [c for c in flags.columns if c in df.columns and c != 'addr']
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        
    df = df.merge(flags, on='addr', how='left')

    return df

In [9]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260329-231025-mesh"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260329-231025-mesh


In [10]:
benchmark = 'syr2k'
trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)
df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.0232067800,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,0.0232067920,571330816,2718494720,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2,0.0232068140,571330816,2718494720,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,0.0232069360,571330816,2718494720,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,3,3
4,0.0232070920,571330816,2718494720,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603034,0.0318025340,571331124,2187653376,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
603035,0.0318026460,571331124,2277272640,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,1
603036,0.0318026730,571331124,2312732864,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,1
603037,0.0318026980,571331132,2420057856,19,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [11]:
rdma_df = df.copy()

for i in range(num_chiplets):
    col = str(i)
    rdma_df[col] = np.where(rdma_df['owner'] == i, 0, rdma_df[col])

df['is_rdma'] = rdma_df[curr_chip_cols].any(axis=1)

In [12]:
df = get_detailed_df(df)

In [13]:
import numpy as np

# 1. 0이 아닌 위치의 불리언 마스크 생성 (NumPy array)
vals = df[curr_chip_cols].values
mask = vals != 0

# 2. 컬럼명 배열 준비
col_names = np.array(curr_chip_cols)

# 3. 각 행별로 마스크를 씌워 리스트 생성 (가장 빠른 파이썬 루프 구조)
df['requestors'] = [col_names[m].tolist() for m in mask]

In [14]:
rdma_df = df[df['is_rdma'] == True]

In [15]:
addr_df = rdma_df.groupby('addr').agg({
    'access_count': 'sum',
    'shared_degree': 'max',
    'owner': 'first',
}).sort_values(by='access_count', ascending=False).reset_index()

hot_threshold = addr_df['access_count'].quantile(0.95)   # 상위 5%
warm_threshold = addr_df['access_count'].quantile(0.75)  # 상위 15%

hot_cachelines = addr_df[(addr_df['access_count'] >= hot_threshold) & 
                         (addr_df['shared_degree'] >= 10)]
warm_cachelines = addr_df[(addr_df['access_count'] >= warm_threshold) &
                          (addr_df['shared_degree'] >= 5) ]

print(f"Total Cachelines: {len(addr_df)}")
print(f"Total Cacheliens Access Sum: {addr_df['access_count'].sum()}")
print(f"Hot Cachelines: {len(hot_cachelines)}")
print(f"Hot Cachelines Access Sum: {hot_cachelines['access_count'].sum()}")
print(f"Hot Cachelines Access Mean: {hot_cachelines['access_count'].mean():.2f}")
print(f"Hot Cachelines Shared Degree Mean: {hot_cachelines['shared_degree'].mean():.2f}")

print(f"Warm Cachelines: {len(warm_cachelines)}")
print(f"Warm Cachelines Access Sum: {warm_cachelines['access_count'].sum()}")
print(f"Warm Cachelines Access Mean: {warm_cachelines['access_count'].mean():.2f}")
print(f"Warm Cachelines Shared Degree Mean: {warm_cachelines['shared_degree'].mean():.2f}")

Total Cachelines: 184753
Total Cacheliens Access Sum: 420234
Hot Cachelines: 2795
Hot Cachelines Access Sum: 206055
Hot Cachelines Access Mean: 73.72
Hot Cachelines Shared Degree Mean: 32.78
Warm Cachelines: 2905
Warm Cachelines Access Sum: 207458
Warm Cachelines Access Mean: 71.41
Warm Cachelines Shared Degree Mean: 31.79


In [16]:
def get_mesh_hop_count(src_node, dst_node, mesh_width=6):
    src_x = src_node % mesh_width
    src_y = src_node // mesh_width
    
    dst_x = dst_node % mesh_width
    dst_y = dst_node // mesh_width
    
    x_hops = abs(src_x - dst_x)
    y_hops = abs(src_y - dst_y)
    
    total_hops = x_hops + y_hops
    
    return total_hops

In [ ]:
import numpy as np
import pandas as pd
import mmh3 # MurmurHash3 (하드웨어 해시와 유사, pip install mmh3 필요)

CBF_SIZE = 1024 * 8 # 슬롯 개수 (예: 8K 슬롯)
COUNTER_BITS = 8  # 각 카운터의 비트 수 (0~255까지 표현)
MAX_COUNT = (2 ** COUNTER_BITS) - 1
NUM_HASHES = 3    # 해시 함수 개수 (k)
HOT_THRESHOLD = 2**(COUNTER_BITS)-1    # Hot으로 판정할 임계값 (접근 횟수)
WARM_THRESHOLD = 2**(COUNTER_BITS-2)-1   # Warm으로 판정할 임계값 (접근 횟수)
DECAY_INTERVAL = 10000 # 시간(cycle) 단위의 간격

cbf = np.zeros(CBF_SIZE, dtype=np.uint8)
cbf_predicted_hot = set()
cbf_predicted_warm = set()

# Target-side: 8-bit Full Counter (0-255)
# uint8 배열로 [NUM_CHIPLETS, CBF_SIZE] 행렬 생성
target_cbfs = np.zeros((num_chiplets, CBF_SIZE), dtype=np.uint8)

# Source-side: 2-bit State Filter (0: Cold, 1: Warm, 2: Hot)
# 메모리 절약을 위해 uint8을 쓰되 값만 0, 1, 2로 관리
source_cbfs = np.zeros((num_chiplets, CBF_SIZE), dtype=np.uint8)

# 각 칩렛의 마지막 Decay 발생 시간 추적
last_decay_times = np.full(num_chiplets, rdma_df['time'].iloc[0])

# 결과 저장을 위한 셋 (각 칩렛별로 식별된 Hot Addr)
detected_hot_cachelines = [set() for _ in range(num_chiplets)]
detected_warm_cachelines = [set() for _ in range(num_chiplets)]

last_decay_time = rdma_df['time'].iloc[0] # 트레이스의 시작 시간으로 초기화
last_decay_time *= 10**9

all_chiplets = list(range(num_chiplets))
hot_cachelines_set = set(hot_cachelines['addr'])
warm_cachelines_set = set(warm_cachelines['addr'])

# --- 2. 에뮬레이션 루프 ---
for _, row in rdma_df.iterrows():
    curr_time = row['time']
    owner = int(row['owner'])
    # src_chiplets가 리스트나 여러 ID의 묶음인 경우를 처리
    src_list = row['requestors'] if isinstance(row['requestors'], list) else [int(row['requestors'])]

    addr = int(row['addr'])
    key_bytes = addr.to_bytes(8, 'little')

    for chip_id in all_chiplets:
        if curr_time - last_decay_times[chip_id] >= DECAY_INTERVAL:
            target_cbfs[chip_id] = target_cbfs[chip_id] >> 1
            last_decay_times[chip_id] = curr_time

    # [Step 2] Target-side Update (단일 목적지)
    # 여러 소스가 동시에 요청하더라도 실제 데이터 소유자는 하나이므로 한 번 업데이트
    # (또는 요청 수만큼 업데이트하고 싶다면 루프 안으로 이동 가능)
    for src in src_list:
        i = get_cluster_id(int(src))
        h_idx = mmh3.hash(key_bytes, i) % CBF_SIZE
        if target_cbfs[owner, h_idx] < MAX_COUNT:
            target_cbfs[owner, h_idx] += 1
    
    hot_count = 0
    warm_count = 0

    owner_cluster_id = get_cluster_id(owner)
    for cls_id in range(NUM_HASHES):
        if cls_id == owner_cluster_id:
            continue
        
        h_idx = mmh3.hash(key_bytes, cls_id) % CBF_SIZE
        if target_cbfs [owner, h_idx] >= HOT_THRESHOLD:
            hot_count += 1
        elif target_cbfs[owner, h_idx] >= WARM_THRESHOLD:
            warm_count += 1

    if hot_count >= 2:
        detected_hot_cachelines[owner].add(addr)
        cbf_predicted_hot.add(addr)
    elif warm_count >= 2:
        detected_warm_cachelines[owner].add(addr)
        cbf_predicted_warm.add(addr)

hot_tp = len(cbf_predicted_hot.intersection(hot_cachelines_set)) # True Positive
hot_fp = len(cbf_predicted_hot - hot_cachelines_set)  # False Positive
hot_fn = len(hot_cachelines_set - cbf_predicted_hot)  # False Negative

hot_precision = hot_tp / (hot_tp + hot_fp) if (hot_tp + hot_fp) > 0 else 0
hot_recall = hot_tp / (hot_tp + hot_fn) if (hot_tp + hot_fn) > 0 else 0

print(f"--- Hot CBF Simulation Results (Size: {CBF_SIZE}, k: {NUM_HASHES}) ---")
print(f"True Hot Addrs (Actual): {len(hot_cachelines_set)}")
print(f"Predicted Hot Addrs (CBF): {len(cbf_predicted_hot)}")
print(f"Precision (정밀도): {hot_precision:.4f} (CBF가 Hot이라고 한 것 중 실제 Hot인 비율)")
print(f"Recall (재현율): {hot_recall:.4f} (실제 Hot 중 CBF가 찾아낸 비율)")
print(f"False Positive Count: {hot_fp} (해시 충돌로 인해 Hot으로 오해한 개수)")

print(f"CBF Predicted Hot Addrs: {len(cbf_predicted_hot)}")
print(f"Top N Actual Hot Addrs: {len(hot_cachelines_set)}")
print(f"Access Count Mean: {addr_df[addr_df['addr'].isin(cbf_predicted_hot)]['access_count'].mean():.2f}")
print(f"Shared Degree Mean: {addr_df[addr_df['addr'].isin(cbf_predicted_hot)]['shared_degree'].mean():.2f}")
print(f"\n")

warm_tp = len(cbf_predicted_warm.intersection(warm_cachelines_set)) # True Positive
warm_fp = len(cbf_predicted_warm - warm_cachelines_set)  # False Positive
warm_fn = len(warm_cachelines_set - cbf_predicted_warm)  # False Negative

warm_precision = warm_tp / (warm_tp + warm_fp) if (warm_tp + warm_fp) > 0 else 0
warm_recall = warm_tp / (warm_tp + warm_fn) if (warm_tp + warm_fn) > 0 else 0

print(f"--- Warm CBF Simulation Results (Size: {CBF_SIZE}, k: {NUM_HASHES}) ---")
print(f"True Warm Addrs (Actual): {len(warm_cachelines_set)}")
print(f"Predicted Warm Addrs (CBF): {len(cbf_predicted_warm)}")
print(f"Precision (정밀도): {warm_precision:.4f} (CBF가 Warm이라고 한 것 중 실제 Warm인 비율)")
print(f"Recall (재현율): {warm_recall:.4f} (실제 Warm 중 CBF가 찾아낸 비율)") 
print(f"False Positive Count: {warm_fp} (해시 충돌로 인해 Warm으로 오해한 개수)")
print(f"Access Count Mean: {addr_df[addr_df['addr'].isin(cbf_predicted_warm)]['access_count'].mean():.2f}")
print(f"Shared Degree Mean: {addr_df[addr_df['addr'].isin(cbf_predicted_warm)]['shared_degree'].mean():.2f}")

--- Hot CBF Simulation Results (Size: 8192, k: 3) ---
True Hot Addrs (Actual): 2795
Predicted Hot Addrs (CBF): 0
Precision (정밀도): 0.0000 (CBF가 Hot이라고 한 것 중 실제 Hot인 비율)
Recall (재현율): 0.0000 (실제 Hot 중 CBF가 찾아낸 비율)
False Positive Count: 0 (해시 충돌로 인해 Hot으로 오해한 개수)
CBF Predicted Hot Addrs: 0
Top N Actual Hot Addrs: 2795
Access Count Mean: nan
Shared Degree Mean: nan


--- Warm CBF Simulation Results (Size: 8192, k: 3) ---
True Warm Addrs (Actual): 2905
Predicted Warm Addrs (CBF): 50
Precision (정밀도): 1.0000 (CBF가 Warm이라고 한 것 중 실제 Warm인 비율)
Recall (재현율): 0.0172 (실제 Warm 중 CBF가 찾아낸 비율)
False Positive Count: 0 (해시 충돌로 인해 Warm으로 오해한 개수)
Access Count Mean: 85.02
Shared Degree Mean: 31.98


In [ ]:
import numpy as np
import pandas as pd
import mmh3
import os

# 1. 이전 단계에서 정의한 벤치마크별 파라미터 딕셔너리 (문자열 형태 복사본 사용)
# 여기에 위에서 드린 params 딕셔너리를 그대로 넣으시면 됩니다.
params = {
    "gups": {"hot_acc": 400, "hot_shr": 35, "warm_acc": 300, "warm_shr": 15},
    "gemm": {"hot_acc": 1500, "hot_shr": 64, "warm_acc": 700, "warm_shr": 32},
    # ... 나머지 앱들도 포함 ...
}

# 공통 하드웨어 설정
CBF_SIZE = 1024 * 8 
COUNTER_BITS = 8
MAX_COUNT = 255
NUM_HASHES = 3
DECAY_INTERVAL = 10000

# 결과 저장을 위한 리스트
all_results = []

# --- 시뮬레이션 메인 루프 ---
# benchmarks 리스트를 순회 (실제 데이터프레임 파일들이 있는 경로를 가정)
for bench_name in benchmarks:
    print(f"\n>>> Processing Benchmark: {bench_name}")
    
    # 1. 데이터 로드 (실제 환경에 맞춰 파일명/경로 수정 필요)
    # 예: rdma_df = pd.read_csv(f"./traces/{bench_name}_trace.csv")
    # (이미 rdma_df가 준비되어 있다면 이 단계는 skip)
    
    # 2. 해당 벤치마크 전용 파라미터 추출
    p = params.get(bench_name, {"hot_acc": 100, "hot_shr": 10, "warm_acc": 50, "warm_shr": 5})
    
    # 정답셋(Ground Truth) 구성 (벤치마크별 Threshold 적용)
    # addr_df 생성 로직이 루프 내부에 있어야 함
    addr_df = rdma_df.groupby('addr').agg({
        'access_count': 'sum',
        'shared_degree': 'max',
        'owner': 'first',
    }).reset_index()

    hot_cachelines_set = set(addr_df[(addr_df['access_count'] >= p['hot_acc']) & 
                                     (addr_df['shared_degree'] >= p['hot_shr'])]['addr'])
    warm_cachelines_set = set(addr_df[(addr_df['access_count'] >= p['warm_acc']) & 
                                      (addr_df['shared_degree'] >= p['warm_shr'])]['addr'])

    # 3. CBF 초기화 및 에뮬레이션 (작성하신 코드 핵심 로직)
    target_cbfs = np.zeros((num_chiplets, CBF_SIZE), dtype=np.uint8)
    last_decay_times = np.full(num_chiplets, rdma_df['time'].iloc[0])
    cbf_predicted_hot = set()
    cbf_predicted_warm = set()

    for _, row in rdma_df.iterrows():
        curr_time = row['time']
        owner = int(row['owner'])
        src_list = row['requestors'] if isinstance(row['requestors'], list) else [int(row['requestors'])]
        addr = int(row['addr'])
        key_bytes = addr.to_bytes(8, 'little')

        # Decay 로직
        for chip_id in range(num_chiplets):
            if curr_time - last_decay_times[chip_id] >= DECAY_INTERVAL:
                target_cbfs[chip_id] >>= 1
                last_decay_times[chip_id] = curr_time

        # Update
        for src in src_list:
            cls_id = get_cluster_id(int(src))
            h_idx = mmh3.hash(key_bytes, cls_id) % CBF_SIZE
            if target_cbfs[owner, h_idx] < MAX_COUNT:
                target_cbfs[owner, h_idx] += 1
        
        # Identification (HOT_THRESHOLD/WARM_THRESHOLD는 8-bit 카운터 포화 기준인 255/63 등 고정값 사용 권장)
        # 벤치마크 스케일과 상관없이 '카운터가 꽉 찼는가'가 중요하기 때문
        hot_count = 0
        warm_count = 0
        for cls_id in range(NUM_HASHES):
            h_idx = mmh3.hash(key_bytes, cls_id) % CBF_SIZE
            if target_cbfs[owner, h_idx] >= 254: # Hot threshold (Saturation)
                hot_count += 1
            elif target_cbfs[owner, h_idx] >= 63: # Warm threshold
                warm_count += 1

        if hot_count >= 2: cbf_predicted_hot.add(addr)
        elif warm_count >= 2: cbf_predicted_warm.add(addr)

    # 4. 결과 지표 계산
    def get_metrics(predicted, actual):
        tp = len(predicted.intersection(actual))
        fp = len(predicted - actual)
        fn = len(actual - predicted)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        return tp, fp, precision, recall

    h_tp, h_fp, h_prec, h_rec = get_metrics(cbf_predicted_hot, hot_cachelines_set)
    w_tp, w_fp, w_prec, w_rec = get_metrics(cbf_predicted_warm, warm_cachelines_set)

    # 5. 리스트에 저장
    all_results.append({
        "Benchmark": bench_name,
        "Actual_Hot": len(hot_cachelines_set),
        "Pred_Hot": len(cbf_predicted_hot),
        "Hot_Precision": h_prec,
        "Hot_Recall": h_rec,
        "Hot_FP": h_fp,
        "Actual_Warm": len(warm_cachelines_set),
        "Pred_Warm": len(cbf_predicted_warm),
        "Warm_Precision": w_prec,
        "Warm_Recall": w_rec,
        "Warm_FP": w_fp,
        "Param_Hot_Acc": p['hot_acc'],
        "Param_Hot_Shr": p['hot_shr']
    })

# --- 최종 결과 CSV 저장 ---
results_df = pd.DataFrame(all_results)
results_df.to_csv("cbf_simulation_summary.csv", index=False)
print("\n[!] All simulations completed. Results saved to 'cbf_simulation_summary.csv'.")